<a href="https://colab.research.google.com/github/Harisudharshan-2005/Hari_summer_internship/blob/main/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install spark

In [ ]:
!pip install pyspark --quiet
print("Pyspark Installed Successfully !!!!!!")

Pyspark Installed Successfully !!!!!!


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder\
        .appName("Day4_Big_Data_Sales")\
        .config('sql.adaptive.enabld','true')\
        .getOrCreate()
print("Spark version : ",spark.version)
print("SparkSession : ACTIVE")
print("Application : ",spark.sparkContext.appName)

Spark version :  4.0.2
SparkSession : ACTIVE
Application :  Day4_Big_Data_Sales


In [ ]:
df_bronze = spark.read\
            .option('header','true')\
            .option('inferSchema','true')\
            .csv("/content/drive/MyDrive/Summer_Internship_2026/large_sales_data.csv")
print("=== DATA SETS LOADED SUCESSFULLY ====")

=== DATA SETS LOADED SUCESSFULLY ====


In [ ]:
print("Number of Rows : ",df_bronze.count())

Number of Rows :  5000


In [ ]:
print("Number of Columns : ",df_bronze.columns)

Number of Columns :  ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']


In [ ]:
print("Number of Columns : ",len(df_bronze.columns))

Number of Columns :  13


In [ ]:
print("=== Schema of Datasets ===")
df_bronze.printSchema()

=== Schema of Datasets ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print("=== Printing First 5 Rows ===")
df_bronze.show(5,truncate=False)

=== Printing First 5 Rows ===
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat   

In [ ]:
print("=== Basic Statistical Columns ===")
df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()

=== Basic Statistical Columns ===
+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+



In [ ]:
print("=== Printing Last 5 Rows ===")
last = df_bronze.tail(5)
spark.createDataFrame(last,df_bronze.schema).show(truncate=False)

=== Printing Last 5 Rows ===
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|5996    |Ananya Das   |Mouse     |Accessories|13      |800       |10400  |2023-11-18|Bangalore|South |Meera Patel|Net Banking     |Cancelled   |
|5997    |Suresh Rao   |Webcam    |Accessories|9       |2500      |22500  |2023-06-07|Chennai  |South |Sunita Rao |Credit Card     |Delivered   |
|5998    |Arjun Nair   |Webcam    |Accessories|1       |2500      |2500   |2023-04-07|Jaipur   |North |Kavya Reddy|Net Banking     |Cancelled   |
|5999    |Arjun Nair   |Laptop    |Electronics|14      |45000     |630000 |2023-07-30|Delhi    

In [ ]:
df_bronze.write\
    .mode('overwrite')\
    .parquet('sales_bronze.parquet')

In [ ]:
import os
def dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total = 0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath,f))
  return total / 1024

csv_size = dir_size('/content/drive/MyDrive/Summer_Internship_2026/large_sales_data.csv')
parquet_size = dir_size('/content/sales_bronze.parquet')
reduction = (1 - parquet_size/csv_size)*100
print("CSV size : ",csv_size,"KB")
print("Parquet size : ",parquet_size,"KB")
print(f"Reduction : {reduction:.1f} % Smaller")
print(f"At 1 TB scale : 1000 GB -> parquet : {1000*(1-reduction/100):.0f} GB")

CSV size :  529.3125 KB
Parquet size :  55.09765625 KB
Reduction : 89.6 % Smaller
At 1 TB scale : 1000 GB -> parquet : 104 GB


In [ ]:
print("=== Basic Statistical Columns ===")
price = df_bronze['unit_price'] + df_bronze['revenue']
df_new = df_bronze.select(price.alias("sum_of_unit_price_and_revenue"))
df_new.show()

In [ ]:
price = df_bronze['unit_price'] + df_bronze['revenue']
df_new = df_bronze.select('unit_price', 'revenue',price.alias("sum_of_unit_price_and_revenue"))
df_new.show()

In [ ]:
#Transformation in spark
# df.select('col1')
# df.filter(condition)
# df.setColumn('new',expr)
# df.groupBy('col1')
# df.orderBy('col1')
# df.drop('col1')
# df.dropDuplicate()

#Actions in spark
#df.show()
#df.count()
#df.collect()

In [ ]:
df_bronze.select('sales_rep').show()

+------------+
|   sales_rep|
+------------+
| Meera Patel|
| Anil Sharma|
| Meera Patel|
|  Ravi Kumar|
|  Sunita Rao|
| Anil Sharma|
|  Priya Nair|
| Meera Patel|
|  Priya Nair|
| Meera Patel|
| Meera Patel|
|Deepak Joshi|
| Suresh Iyer|
|Deepak Joshi|
|  Sunita Rao|
|  Sunita Rao|
| Anil Sharma|
| Kavya Reddy|
|Deepak Joshi|
| Anil Sharma|
+------------+
only showing top 20 rows


In [ ]:
df_bronze.filter(df_bronze['revenue']>10000).select('revenue').show()

+-------+
|revenue|
+-------+
| 264000|
| 120000|
| 160000|
|  14000|
|  25000|
| 585000|
|  38500|
|  35000|
|  10400|
|  18000|
|  14000|
|  15600|
|  40500|
|  12500|
| 108000|
|  22500|
| 675000|
|  10800|
|  38500|
|  88000|
+-------+
only showing top 20 rows


In [ ]:
# Group by 'region' and sum 'revenue', then order by total revenue
print("=== Revenue by Region (Grouped and Ordered) ===")
df_bronze.groupBy('region')\
    .agg(F.sum('revenue').alias('total_revenue'))\
    .orderBy('total_revenue', ascending=False)\
    .show()

=== Revenue by Region (Grouped and Ordered) ===
+------+-------------+
|region|total_revenue|
+------+-------------+
|  West|    198275600|
| South|    147145900|
| North|     99878400|
|  East|     50547700|
+------+-------------+



In [ ]:
print("=== Grouping and Ordering the datasets ===")
df_bronze.groupBy('category')\
    .agg(F.sum('revenue').alias('total_revenue'))\
    .orderBy('total_revenue',ascending = False)\
    .show()

In [ ]:
df_silver = df_bronze.select('product', 'revenue', 'order_id') \
                     .dropDuplicates() \
                     .dropna(subset=['product', 'revenue', 'order_id'])

In [ ]:
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'),'yyyy-MM-dd')
).select('customer_name', 'order_id', 'order_date')
df_silver.show()

+-------------+--------+----------+
|customer_name|order_id|order_date|
+-------------+--------+----------+
|  Divya Singh|    1246|2023-02-07|
|   Ananya Das|    1323|2023-01-24|
|Kavya Nambiar|    1386|2023-04-16|
|   Amit Verma|    1939|2023-12-21|
|  Sneha Reddy|    2222|2023-08-23|
|  Pooja Gupta|    2441|2023-05-26|
|  Meera Joshi|    2448|2023-11-04|
|   Arjun Nair|    2477|2023-01-24|
|   Ananya Das|    2478|2023-06-17|
|  Priya Patel|    2696|2023-06-18|
|   Ananya Das|    2794|2023-08-24|
|  Kiran Kumar|    3255|2023-07-30|
|  Priya Patel|    3475|2023-08-29|
|  Meera Joshi|    3920|2023-11-02|
|  Meera Joshi|    4353|2023-01-23|
|   Ananya Das|    4363|2023-08-14|
|Kavya Nambiar|    4979|2023-04-11|
|  Pooja Gupta|    5049|2023-03-14|
|Kavya Nambiar|    5216|2023-09-12|
|  Vikram Iyer|    5284|2023-07-27|
+-------------+--------+----------+
only showing top 20 rows


In [ ]:
df_silver = df_bronze.dropDuplicates().dropna(subset=['order_id', 'product', 'revenue'])

df_silver = df_silver.withColumn(
    'revenue',
    spark_round(col('revenue'),2)
).select('order_id','customer_name','revenue')

df_silver.show()

+--------+-------------+-------+
|order_id|customer_name|revenue|
+--------+-------------+-------+
|    1246|  Divya Singh|  13200|
|    1323|   Ananya Das|  17500|
|    1386|Kavya Nambiar|  58500|
|    1939|   Amit Verma|   9600|
|    2222|  Sneha Reddy| 180000|
|    2441|  Pooja Gupta|  38500|
|    2448|  Meera Joshi|  35000|
|    2477|   Arjun Nair| 360000|
|    2478|   Ananya Das| 320000|
|    2696|  Priya Patel| 225000|
|    2794|   Ananya Das|   6400|
|    3255|  Kiran Kumar| 132000|
|    3475|  Priya Patel| 315000|
|    3920|  Meera Joshi| 330000|
|    4353|  Meera Joshi|  10500|
|    4363|   Ananya Das| 405000|
|    4979|Kavya Nambiar|  15000|
|    5049|  Pooja Gupta|   1800|
|    5216|Kavya Nambiar| 264000|
|    5284|  Vikram Iyer| 450000|
+--------+-------------+-------+
only showing top 20 rows


In [ ]:
df_silver.write\
          .mode('overwrite')\
          .parquet('sales_silver.parquet')
print("Silver Parquet Saved : sales_silver.parquet")
print(f"Silver size : {dir_size("sales_silver.parquet"):.2f} KB")
df_verify = spark.read.parquet('sales_silver.parquet')
print(f"Read Back Rows : {df_verify.count()} (should match the Silver count)")
df_verify.show()
print("=== Printing Schema ===")
df_verify.printSchema()

Silver Parquet Saved : sales_silver.parquet
Silver size : 30.51 KB
Read Back Rows : 5000 (should match the Silver count)
+-------------+--------+----------+
|customer_name|order_id|order_date|
+-------------+--------+----------+
|  Divya Singh|    1246|2023-02-07|
|   Ananya Das|    1323|2023-01-24|
|Kavya Nambiar|    1386|2023-04-16|
|   Amit Verma|    1939|2023-12-21|
|  Sneha Reddy|    2222|2023-08-23|
|  Pooja Gupta|    2441|2023-05-26|
|  Meera Joshi|    2448|2023-11-04|
|   Arjun Nair|    2477|2023-01-24|
|   Ananya Das|    2478|2023-06-17|
|  Priya Patel|    2696|2023-06-18|
|   Ananya Das|    2794|2023-08-24|
|  Kiran Kumar|    3255|2023-07-30|
|  Priya Patel|    3475|2023-08-29|
|  Meera Joshi|    3920|2023-11-02|
|  Meera Joshi|    4353|2023-01-23|
|   Ananya Das|    4363|2023-08-14|
|Kavya Nambiar|    4979|2023-04-11|
|  Pooja Gupta|    5049|2023-03-14|
|Kavya Nambiar|    5216|2023-09-12|
|  Vikram Iyer|    5284|2023-07-27|
+-------------+--------+----------+
only showing to

In [ ]:
df_silver = df_bronze.select('product', 'revenue', 'order_id') \
                     .dropDuplicates() \
                     .dropna(subset=['product', 'revenue', 'order_id'])

top_products = df_silver\
              .groupBy('product')\
              .agg(F.sum('revenue').alias('total_revenue'),
                   F.count('order_id').alias('num_order'),
                   F.avg('revenue').alias('avg_order_revenue')
              )\
              .orderBy('total_revenue',ascending=False)
print("=== Top 5 Rows ===")
top_products.show(5,truncate = False)

=== Top 5 Rows ===
+-------+-------------+---------+------------------+
|product|total_revenue|num_order|avg_order_revenue |
+-------+-------------+---------+------------------+
|Laptop |182700000    |502      |363944.22310756973|
|Tablet |135104000    |532      |253954.8872180451 |
|Monitor|82126000     |481      |170740.12474012474|
|Printer|44544000     |488      |91278.68852459016 |
|Speaker|16317000     |470      |34717.02127659575 |
+-------+-------------+---------+------------------+
only showing top 5 rows


In [ ]:
answer = df_bronze\
         .groupBy('region')\
         .agg(F.sum('revenue').alias('total_revenue'),
              F.count('order_id').alias('total_order'))\
         .orderBy(F.col('total_revenue').desc()).select('region','total_revenue','total_order')
answer.show()

+------+-------------+-----------+
|region|total_revenue|total_order|
+------+-------------+-----------+
|  West|    198275600|       2021|
| South|    147145900|       1483|
| North|     99878400|        995|
|  East|     50547700|        501|
+------+-------------+-----------+



In [ ]:
# Ensure 'order_date', 'revenue', and 'order_id' are available for monthly analysis.
# We create a temporary DataFrame for this specific calculation from df_bronze.
df_monthly_analysis = df_bronze.select('order_date', 'revenue', 'order_id') \
                               .filter(F.col('order_date').isNotNull()) \
                               .filter(F.col('revenue').isNotNull()) \
                               .dropDuplicates()

df_month = df_monthly_analysis\
          .groupBy(month('order_date').alias('order_month'))\
          .agg(F.sum('revenue').alias('Monthly_revenue'),
               F.count('order_id').alias('Monthly_order'))\
          .orderBy(F.col('order_month'),ascending=True).select('order_month','Monthly_revenue','Monthly_order')
df_month.show()

+-----------+---------------+-------------+
|order_month|Monthly_revenue|Monthly_order|
+-----------+---------------+-------------+
|          1|       41068200|          423|
|          2|       34485400|          375|
|          3|       40031200|          451|
|          4|       38857100|          390|
|          5|       39984500|          423|
|          6|       40707400|          390|
|          7|       42640700|          405|
|          8|       43718500|          418|
|          9|       37640200|          398|
|         10|       47839000|          479|
|         11|       44577100|          419|
|         12|       44298300|          429|
+-----------+---------------+-------------+



In [ ]:
df_monthly_analysis = df_bronze.select('order_date', 'revenue', 'order_id') \
                               .filter(F.col('order_date').isNotNull()) \
                               .filter(F.col('revenue').isNotNull()) \
                               .dropDuplicates()

df_month = df_monthly_analysis\
          .groupBy(F.month('order_date').alias('month_num'), F.date_format(F.col('order_date'), 'MMMM').alias('order_month'))\
          .agg(F.sum('revenue').alias('Monthly_revenue'),
               F.count('order_id').alias('Monthly_order'))\
          .orderBy(F.col('month_num'), ascending=True)\
          .select('order_month','Monthly_revenue','Monthly_order')
df_month.show()

+-----------+---------------+-------------+
|order_month|Monthly_revenue|Monthly_order|
+-----------+---------------+-------------+
|    January|       41068200|          423|
|   February|       34485400|          375|
|      March|       40031200|          451|
|      April|       38857100|          390|
|        May|       39984500|          423|
|       June|       40707400|          390|
|       July|       42640700|          405|
|     August|       43718500|          418|
|  September|       37640200|          398|
|    October|       47839000|          479|
|   November|       44577100|          419|
|   December|       44298300|          429|
+-----------+---------------+-------------+

